# Moduł 11: Testing - Testowanie API z pytest

**Ćwiczenie:** #12 - Testing

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [FastAPI TestClient](#2-fastapi-testclient)
3. [Pytest Fixtures](#3-pytest-fixtures)
4. [Testing CRUD Operations](#4-testing-crud-operations)
5. [Testing Authentication](#5-testing-authentication)
6. [Dependency Overrides](#6-dependency-overrides)
7. [Parametrized Tests](#7-parametrized-tests)
8. [Test Organization](#8-test-organization)
9. [Best Practices](#9-best-practices)
10. [Demo Live Coding](#10-demo-live-coding)
11. [Przygotowanie do ćwiczenia](#11-przygotowanie-do-ćwiczenia)
12. [Podsumowanie](#12-podsumowanie)

---

## 1. Wprowadzenie

### Dlaczego testujemy API?

**Testy automatyczne są kluczowe:**
1. **Wykrywanie błędów wcześnie** - przed produkcją
2. **Dokumentacja żywa** - testy pokazują jak API działa
3. **Refactoring bez strachu** - możesz zmieniać kod
4. **Regression prevention** - upewniam się że bug nie wróci
5. **CI/CD** - automatyczne testy przy każdym commit

### Pytest vs unittest

**unittest (built-in):**
```python
import unittest

class TestAPI(unittest.TestCase):
    def test_get_posts(self):
        response = client.get("/posts")
        self.assertEqual(response.status_code, 200)
```

**pytest (recommended):**
```python
def test_get_posts():
    response = client.get("/posts")
    assert response.status_code == 200
```

**Dlaczego pytest?**
- ✅ Prostszy syntax (`assert` zamiast `self.assertEqual`)
- ✅ Fixtures zamiast setUp/tearDown
- ✅ Lepsze error messages
- ✅ Parametrized tests
- ✅ Bogaty ekosystem pluginów

### AAA Pattern

**Każdy test składa się z 3 części:**

```python
def test_create_post():
    # ARRANGE (przygotuj dane)
    post_data = {"title": "Test", "content": "Hello"}
    
    # ACT (wykonaj akcję)
    response = client.post("/posts", json=post_data)
    
    # ASSERT (sprawdź wynik)
    assert response.status_code == 201
    assert response.json()["title"] == "Test"
```

---

## 2. FastAPI TestClient

### Czym jest TestClient?

**TestClient** - narzędzie FastAPI do testowania bez uruchamiania serwera:
- Symuluje HTTP requests
- Zwraca Response objects
- Nie wymaga `uvicorn`
- Szybkie i deterministyczne

**Instalacja:**
```bash
pip install pytest httpx
```

### Podstawowe użycie

In [ ]:
from fastapi.testclient import TestClient
from main import app  # Twoja aplikacja FastAPI

client = TestClient(app)

def test_read_root():
    """Test GET /"""
    # ACT
    response = client.get("/")
    
    # ASSERT
    assert response.status_code == 200
    assert response.json() == {"message": "Hello World"}

### HTTP Methods

In [ ]:
# GET
response = client.get("/posts")
response = client.get("/posts/1")
response = client.get("/posts?page=1&size=10")  # Query params

# POST (JSON body)
response = client.post("/posts", json={"title": "Test", "content": "Hello"})

# POST (Form data - dla OAuth2)
response = client.post("/auth/login", data={"username": "john", "password": "secret"})

# PUT
response = client.put("/posts/1", json={"title": "Updated"})

# DELETE
response = client.delete("/posts/1")

# Headers (np. Authorization)
response = client.get("/me", headers={"Authorization": "Bearer <token>"})

### Response object

In [ ]:
response = client.get("/posts/1")

# Status code
assert response.status_code == 200
assert response.status_code == 404

# JSON body
data = response.json()
assert data["id"] == 1
assert data["title"] == "Test"

# Headers
assert response.headers["content-type"] == "application/json"

# Text (raw)
text = response.text

---

## 3. Pytest Fixtures

### Czym są fixtures?

**Fixtures** - funkcje dostarczające dane/setup dla testów:
- Reusable setup code
- Automatyczne cleanup
- Dependency injection dla testów

### Fixture: TestClient

In [ ]:
import pytest
from fastapi.testclient import TestClient
from main import app

@pytest.fixture
def client():
    """Fixture dostarczająca TestClient"""
    return TestClient(app)

# Użycie w teście
def test_get_posts(client):  # ← client z fixture
    response = client.get("/posts")
    assert response.status_code == 200

### Fixture: Test Database

**Problem:** Testy nie powinny modyfikować production database!

**Rozwiązanie:** Osobna test database (SQLite in-memory)

In [ ]:
import pytest
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from database import Base

@pytest.fixture
def db_engine():
    """Fixture: Test database engine (SQLite in-memory)"""
    # In-memory SQLite - szybka, usuwa się po teście
    engine = create_engine("sqlite:///:memory:", echo=False)
    
    # Utwórz tabele
    Base.metadata.create_all(bind=engine)
    
    yield engine
    
    # Cleanup (automatycznie po teście)
    Base.metadata.drop_all(bind=engine)

@pytest.fixture
def db_session(db_engine):
    """Fixture: Test database session"""
    SessionLocal = sessionmaker(bind=db_engine)
    session = SessionLocal()
    
    yield session
    
    # Cleanup
    session.close()

### Fixture: Sample Data

In [ ]:
@pytest.fixture
def sample_user(db_session):
    """Fixture: Stwórz przykładowego usera"""
    from models import User
    from auth import hash_password
    
    user = User(
        username="testuser",
        email="test@example.com",
        password_hash=hash_password("testpass123")
    )
    db_session.add(user)
    db_session.commit()
    db_session.refresh(user)
    
    return user

# Użycie
def test_get_user(client, sample_user):
    """Test GET /users/{id}"""
    response = client.get(f"/users/{sample_user.id}")
    assert response.status_code == 200
    assert response.json()["username"] == "testuser"

### Fixture Scopes

**Scope określa jak długo fixture żyje:**

| Scope | Czas życia | Użycie |
|-------|------------|--------|
| **function** | Jedna funkcja testowa (default) | Reset stanu między testami |
| **class** | Wszystkie testy w klasie | Testy grupowane |
| **module** | Wszystkie testy w pliku | Wolne setup (database) |
| **session** | Cała sesja testowa | Bardzo wolne setup |

```python
@pytest.fixture(scope="function")  # Default - nowy client dla każdego testu
def client():
    return TestClient(app)

@pytest.fixture(scope="module")  # Jeden raz per plik
def db_engine():
    engine = create_engine("sqlite:///:memory:")
    return engine
```

---

## 4. Testing CRUD Operations

### Test CREATE (POST)

In [ ]:
def test_create_task(client):
    """
    Test POST /tasks - stwórz nowy task
    """
    # ARRANGE
    task_data = {"name": "Buy milk"}
    
    # ACT
    response = client.post("/tasks", json=task_data)
    
    # ASSERT
    assert response.status_code == 201
    
    data = response.json()
    assert data["name"] == "Buy milk"
    assert "id" in data
    assert isinstance(data["id"], int)

### Test READ (GET)

In [ ]:
def test_get_tasks_empty(client):
    """Test GET /tasks - pusta lista"""
    response = client.get("/tasks")
    
    assert response.status_code == 200
    assert response.json() == []

def test_get_task_by_id(client, sample_task):
    """Test GET /tasks/{id} - pojedynczy task"""
    response = client.get(f"/tasks/{sample_task.id}")
    
    assert response.status_code == 200
    data = response.json()
    assert data["id"] == sample_task.id
    assert data["name"] == sample_task.name

def test_get_task_not_found(client):
    """Test GET /tasks/999 - 404"""
    response = client.get("/tasks/999")
    
    assert response.status_code == 404
    assert "not found" in response.json()["detail"].lower()

### Test UPDATE (PUT)

In [ ]:
def test_update_task(client, sample_task):
    """Test PUT /tasks/{id} - zaktualizuj task"""
    # ARRANGE
    update_data = {"name": "Buy bread"}
    
    # ACT
    response = client.put(f"/tasks/{sample_task.id}", json=update_data)
    
    # ASSERT
    assert response.status_code == 200
    data = response.json()
    assert data["name"] == "Buy bread"
    assert data["id"] == sample_task.id

def test_update_task_not_found(client):
    """Test PUT /tasks/999 - 404"""
    response = client.put("/tasks/999", json={"name": "Updated"})
    assert response.status_code == 404

### Test DELETE

In [ ]:
def test_delete_task(client, sample_task):
    """Test DELETE /tasks/{id}"""
    # ACT
    response = client.delete(f"/tasks/{sample_task.id}")
    
    # ASSERT
    assert response.status_code == 204
    
    # Sprawdź że task nie istnieje
    get_response = client.get(f"/tasks/{sample_task.id}")
    assert get_response.status_code == 404

def test_delete_task_not_found(client):
    """Test DELETE /tasks/999 - 404"""
    response = client.delete("/tasks/999")
    assert response.status_code == 404

---

## 5. Testing Authentication

### Test Login

In [ ]:
def test_login_success(client, sample_user):
    """Test POST /auth/login - successful login"""
    # ARRANGE
    login_data = {
        "username": "testuser",
        "password": "testpass123"
    }
    
    # ACT
    response = client.post("/auth/login", data=login_data)  # Form data!
    
    # ASSERT
    assert response.status_code == 200
    data = response.json()
    assert "access_token" in data
    assert data["token_type"] == "bearer"

def test_login_invalid_password(client, sample_user):
    """Test POST /auth/login - wrong password"""
    response = client.post("/auth/login", data={
        "username": "testuser",
        "password": "wrongpassword"
    })
    
    assert response.status_code == 401

def test_login_user_not_found(client):
    """Test POST /auth/login - user doesn't exist"""
    response = client.post("/auth/login", data={
        "username": "nonexistent",
        "password": "password"
    })
    
    assert response.status_code == 401

### Test Protected Endpoints

In [ ]:
@pytest.fixture
def auth_headers(client, sample_user):
    """Fixture: Headers z JWT tokenem"""
    # Login
    response = client.post("/auth/login", data={
        "username": "testuser",
        "password": "testpass123"
    })
    token = response.json()["access_token"]
    
    # Return headers
    return {"Authorization": f"Bearer {token}"}

def test_get_me_success(client, auth_headers):
    """Test GET /me - authenticated"""
    response = client.get("/me", headers=auth_headers)
    
    assert response.status_code == 200
    data = response.json()
    assert data["username"] == "testuser"

def test_get_me_no_token(client):
    """Test GET /me - brak tokenu (401)"""
    response = client.get("/me")
    assert response.status_code == 401

def test_get_me_invalid_token(client):
    """Test GET /me - nieprawidłowy token (401)"""
    response = client.get("/me", headers={"Authorization": "Bearer invalid_token"})
    assert response.status_code == 401

### Test Authorization (Roles)

In [ ]:
@pytest.fixture
def admin_headers(client, db_session):
    """Fixture: Headers dla admina"""
    from models import User, UserRole
    from auth import hash_password
    
    # Stwórz admina
    admin = User(
        username="admin",
        password_hash=hash_password("admin123"),
        role=UserRole.ADMIN
    )
    db_session.add(admin)
    db_session.commit()
    
    # Login
    response = client.post("/auth/login", data={
        "username": "admin",
        "password": "admin123"
    })
    token = response.json()["access_token"]
    
    return {"Authorization": f"Bearer {token}"}

def test_admin_endpoint_as_user(client, auth_headers):
    """Test GET /admin/stats - user próbuje (403)"""
    response = client.get("/admin/stats", headers=auth_headers)
    assert response.status_code == 403

def test_admin_endpoint_as_admin(client, admin_headers):
    """Test GET /admin/stats - admin (200)"""
    response = client.get("/admin/stats", headers=admin_headers)
    assert response.status_code == 200

---

## 6. Dependency Overrides

### Problem: Test Database

**Aplikacja używa production database:**
```python
# main.py
def get_db():
    db = SessionLocal()  # Production database!
    yield db
    db.close()
```

**Chcemy w testach użyć test database:**

### Rozwiązanie: app.dependency_overrides

In [ ]:
import pytest
from fastapi.testclient import TestClient
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from main import app
from database import get_db, Base

@pytest.fixture
def test_db():
    """Fixture: Test database"""
    # In-memory SQLite
    engine = create_engine("sqlite:///:memory:")
    TestingSessionLocal = sessionmaker(bind=engine)
    
    # Create tables
    Base.metadata.create_all(bind=engine)
    
    # Override dependency
    def override_get_db():
        db = TestingSessionLocal()
        try:
            yield db
        finally:
            db.close()
    
    app.dependency_overrides[get_db] = override_get_db
    
    yield TestingSessionLocal
    
    # Cleanup
    app.dependency_overrides.clear()
    Base.metadata.drop_all(bind=engine)

@pytest.fixture
def client(test_db):
    """Fixture: TestClient z test database"""
    return TestClient(app)

# Teraz wszystkie testy używają test database!
def test_create_task(client):
    response = client.post("/tasks", json={"name": "Test"})
    assert response.status_code == 201

### Override Auth Dependency (Mock User)

In [ ]:
from dependencies import get_current_user
from models import User

@pytest.fixture
def mock_user():
    """Fixture: Mock current user (bez tokenu!)"""
    fake_user = User(id=1, username="testuser", role="user")
    
    def override_get_current_user():
        return fake_user
    
    app.dependency_overrides[get_current_user] = override_get_current_user
    
    yield fake_user
    
    app.dependency_overrides.clear()

# Test bez prawdziwego JWT tokenu!
def test_get_me_mocked(client, mock_user):
    """Test GET /me - bez prawdziwego tokenu"""
    response = client.get("/me")  # Brak Authorization header!
    
    assert response.status_code == 200
    assert response.json()["username"] == "testuser"

---

## 7. Parametrized Tests

### @pytest.mark.parametrize

**Problem:** Testowanie wielu przypadków (duplikacja kodu)

**Rozwiązanie:** Parametrized tests

In [ ]:
import pytest

@pytest.mark.parametrize("task_name, expected_status", [
    ("Valid task", 201),          # Valid
    ("A", 422),                   # Too short (min_length=3)
    ("A" * 101, 422),             # Too long (max_length=100)
    ("", 422),                    # Empty
])
def test_create_task_validation(client, task_name, expected_status):
    """Test walidacji task name"""
    response = client.post("/tasks", json={"name": task_name})
    assert response.status_code == expected_status

# Pytest uruchomi ten test 4 razy z różnymi parametrami!

### Multiple Parameters

In [ ]:
@pytest.mark.parametrize("username, password, expected_status, expected_detail", [
    # Valid credentials
    ("testuser", "testpass123", 200, None),
    
    # Invalid password
    ("testuser", "wrongpass", 401, "Incorrect username or password"),
    
    # User doesn't exist
    ("nonexistent", "password", 401, "Incorrect username or password"),
    
    # Empty fields
    ("", "", 422, None),  # Validation error
])
def test_login_scenarios(
    client,
    sample_user,
    username,
    password,
    expected_status,
    expected_detail
):
    """Test różnych scenariuszy logowania"""
    response = client.post("/auth/login", data={
        "username": username,
        "password": password
    })
    
    assert response.status_code == expected_status
    
    if expected_detail:
        assert expected_detail in response.json()["detail"]

---

## 8. Test Organization

### Struktura projektu

In [ ]:
# project/
# ├── app/
# │   ├── main.py
# │   ├── models.py
# │   ├── database.py
# │   └── dependencies.py
# ├── tests/
# │   ├── conftest.py          # Fixtures globalne
# │   ├── test_auth.py         # Testy autentykacji
# │   ├── test_tasks.py        # Testy CRUD tasks
# │   └── test_users.py        # Testy CRUD users
# ├── pytest.ini               # Konfiguracja pytest
# └── requirements.txt

### conftest.py - Shared Fixtures

**conftest.py** - plik specjalny dla pytest, zawiera fixtures dostępne we wszystkich testach:

In [ ]:
# tests/conftest.py
import pytest
from fastapi.testclient import TestClient
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from main import app
from database import get_db, Base

@pytest.fixture(scope="function")
def test_db():
    """Test database fixture"""
    engine = create_engine("sqlite:///:memory:")
    TestingSessionLocal = sessionmaker(bind=engine)
    Base.metadata.create_all(bind=engine)
    
    def override_get_db():
        db = TestingSessionLocal()
        try:
            yield db
        finally:
            db.close()
    
    app.dependency_overrides[get_db] = override_get_db
    yield TestingSessionLocal
    app.dependency_overrides.clear()
    Base.metadata.drop_all(bind=engine)

@pytest.fixture
def client(test_db):
    """TestClient fixture"""
    return TestClient(app)

@pytest.fixture
def sample_user(test_db):
    """Sample user fixture"""
    from models import User
    from auth import hash_password
    
    db = test_db()
    user = User(
        username="testuser",
        password_hash=hash_password("testpass123")
    )
    db.add(user)
    db.commit()
    db.refresh(user)
    return user

### pytest.ini - Configuration

In [ ]:
# pytest.ini
[pytest]
testpaths = tests
python_files = test_*.py
python_classes = Test*
python_functions = test_*
addopts = 
    -v
    --tb=short
    --strict-markers
markers =
    slow: marks tests as slow (deselect with '-m "not slow"')
    integration: marks tests as integration tests

### Running Tests

```bash
# Wszystkie testy
pytest

# Verbose output
pytest -v

# Konkretny plik
pytest tests/test_auth.py

# Konkretny test
pytest tests/test_auth.py::test_login_success

# Z coverage
pytest --cov=app --cov-report=html

# Stop at first failure
pytest -x

# Run only failed tests from last run
pytest --lf
```

---

## 9. Best Practices

### ✅ Rób tak:

**1. Używaj AAA pattern:**
```python
def test_create_post():
    # ARRANGE
    data = {"title": "Test"}
    
    # ACT
    response = client.post("/posts", json=data)
    
    # ASSERT
    assert response.status_code == 201
```

**2. Testuj różne scenariusze:**
```python
# ✅ Dobre - success + error cases
def test_get_task_success(client, sample_task): pass
def test_get_task_not_found(client): pass
def test_get_task_invalid_id(client): pass
```

**3. Używaj fixtures dla reusable setup:**
```python
# ✅ Dobre - fixture
@pytest.fixture
def auth_headers(client, sample_user):
    # Login logic
    return headers

# ❌ Złe - duplikacja
def test_1():
    # Login logic...
    pass

def test_2():
    # Login logic... (again!)
    pass
```

**4. Test database isolation:**
```python
# ✅ Dobre - in-memory SQLite per test
@pytest.fixture(scope="function")
def test_db():
    engine = create_engine("sqlite:///:memory:")
    # ...

# ❌ Złe - shared database
# Testy wpływają na siebie!
```

**5. Opisowe nazwy testów:**
```python
# ✅ Dobre - jasne co testujemy
def test_create_task_with_valid_data_returns_201(): pass
def test_create_task_with_empty_name_returns_422(): pass

# ❌ Złe - niejasne
def test_1(): pass
def test_task(): pass
```

**6. Testuj edge cases:**
```python
# ✅ Dobre
@pytest.mark.parametrize("name", [
    "",              # Empty
    "A",            # Too short
    "A" * 101,      # Too long
    "Valid name",   # OK
])
def test_task_name_validation(client, name): pass
```

### ❌ Nie rób tak:

**1. Nie używaj production database:**
```python
# ❌ BARDZO ŹLE - modyfikujesz production data!
# Brak dependency_overrides
```

**2. Nie testuj implementation details:**
```python
# ❌ Złe - testuje jak działa, nie co robi
def test_create_task():
    assert "db.add() was called"  # Implementation detail

# ✅ Dobre - testuje rezultat
def test_create_task():
    response = client.post("/tasks", json={"name": "Test"})
    assert response.status_code == 201
```

**3. Nie rób sleep() w testach:**
```python
# ❌ Złe - wolne testy
import time
def test_something():
    time.sleep(5)  # Czekam aż coś się stanie
    # ...

# ✅ Dobre - synchroniczne testy
def test_something():
    # TestClient jest synchroniczny
    response = client.post(...)  # Natychmiast
```

**4. Nie ignoruj warnings:**
```python
# ❌ Złe
import warnings
warnings.filterwarnings("ignore")

# ✅ Dobre - fix warnings!
```

In [ ]:
# tests/conftest.py
import pytest
from fastapi.testclient import TestClient
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from main import app
from database import get_db, Base
from models import Task

@pytest.fixture(scope="function")
def test_db():
    engine = create_engine("sqlite:///:memory:", echo=False)
    TestingSessionLocal = sessionmaker(bind=engine)
    Base.metadata.create_all(bind=engine)
    
    def override_get_db():
        db = TestingSessionLocal()
        try:
            yield db
        finally:
            db.close()
    
    app.dependency_overrides[get_db] = override_get_db
    yield TestingSessionLocal
    app.dependency_overrides.clear()
    Base.metadata.drop_all(bind=engine)

@pytest.fixture
def client(test_db):
    return TestClient(app)

@pytest.fixture
def sample_task(test_db):
    db = test_db()
    task = Task(name="Sample task")
    db.add(task)
    db.commit()
    db.refresh(task)
    return task

**tests/test_tasks.py:**

In [ ]:
# tests/test_tasks.py
import pytest

class TestCreateTask:
    """Tests for POST /tasks"""
    
    def test_create_task_success(self, client):
        """Test creating a task with valid data"""
        response = client.post("/tasks", json={"name": "Buy milk"})
        
        assert response.status_code == 201
        data = response.json()
        assert data["name"] == "Buy milk"
        assert "id" in data
    
    @pytest.mark.parametrize("invalid_name", [
        "",           # Empty
        "A",         # Too short
        "A" * 101,   # Too long
    ])
    def test_create_task_invalid_name(self, client, invalid_name):
        """Test creating task with invalid name"""
        response = client.post("/tasks", json={"name": invalid_name})
        assert response.status_code == 422

class TestGetTasks:
    """Tests for GET /tasks"""
    
    def test_get_tasks_empty(self, client):
        """Test getting tasks when database is empty"""
        response = client.get("/tasks")
        
        assert response.status_code == 200
        assert response.json() == []
    
    def test_get_tasks_with_data(self, client, sample_task):
        """Test getting tasks when database has data"""
        response = client.get("/tasks")
        
        assert response.status_code == 200
        data = response.json()
        assert len(data) == 1
        assert data[0]["name"] == "Sample task"

class TestGetTask:
    """Tests for GET /tasks/{id}"""
    
    def test_get_task_success(self, client, sample_task):
        """Test getting existing task"""
        response = client.get(f"/tasks/{sample_task.id}")
        
        assert response.status_code == 200
        data = response.json()
        assert data["id"] == sample_task.id
        assert data["name"] == "Sample task"
    
    def test_get_task_not_found(self, client):
        """Test getting non-existent task"""
        response = client.get("/tasks/999")
        
        assert response.status_code == 404
        assert "not found" in response.json()["detail"].lower()

class TestUpdateTask:
    """Tests for PUT /tasks/{id}"""
    
    def test_update_task_success(self, client, sample_task):
        """Test updating existing task"""
        response = client.put(
            f"/tasks/{sample_task.id}",
            json={"name": "Updated task"}
        )
        
        assert response.status_code == 200
        data = response.json()
        assert data["name"] == "Updated task"
        assert data["id"] == sample_task.id
    
    def test_update_task_not_found(self, client):
        """Test updating non-existent task"""
        response = client.put("/tasks/999", json={"name": "Updated"})
        assert response.status_code == 404

class TestDeleteTask:
    """Tests for DELETE /tasks/{id}"""
    
    def test_delete_task_success(self, client, sample_task):
        """Test deleting existing task"""
        response = client.delete(f"/tasks/{sample_task.id}")
        
        assert response.status_code == 204
        
        # Verify task is gone
        get_response = client.get(f"/tasks/{sample_task.id}")
        assert get_response.status_code == 404
    
    def test_delete_task_not_found(self, client):
        """Test deleting non-existent task"""
        response = client.delete("/tasks/999")
        assert response.status_code == 404

**Running:**
```bash
pytest tests/test_tasks.py -v

# Output:
# tests/test_tasks.py::TestCreateTask::test_create_task_success PASSED
# tests/test_tasks.py::TestCreateTask::test_create_task_invalid_name[...] PASSED
# tests/test_tasks.py::TestGetTasks::test_get_tasks_empty PASSED
# tests/test_tasks.py::TestGetTasks::test_get_tasks_with_data PASSED
# tests/test_tasks.py::TestGetTask::test_get_task_success PASSED
# tests/test_tasks.py::TestGetTask::test_get_task_not_found PASSED
# tests/test_tasks.py::TestUpdateTask::test_update_task_success PASSED
# tests/test_tasks.py::TestUpdateTask::test_update_task_not_found PASSED
# tests/test_tasks.py::TestDeleteTask::test_delete_task_success PASSED
# tests/test_tasks.py::TestDeleteTask::test_delete_task_not_found PASSED
#
# ==================== 13 passed in 0.15s ====================
```

---

## 12. Podsumowanie

### Kluczowe zagadnienia:

1. **FastAPI TestClient** - testowanie bez serwera
2. **Pytest** - prostszy od unittest
3. **Fixtures** - reusable setup (test_db, client, sample_user)
4. **AAA Pattern** - Arrange, Act, Assert
5. **Dependency Overrides** - test database zamiast production
6. **Parametrized Tests** - testowanie wielu przypadków
7. **Test Organization** - conftest.py, pytest.ini

### Test structure:

```python
# conftest.py
@pytest.fixture
def client(test_db):
    return TestClient(app)

# test_tasks.py
def test_create_task(client):
    # ARRANGE
    data = {"name": "Test"}
    
    # ACT
    response = client.post("/tasks", json=data)
    
    # ASSERT
    assert response.status_code == 201
    assert response.json()["name"] == "Test"
```

### Running tests:
```bash
pytest                          # All tests
pytest -v                       # Verbose
pytest --cov=app                # Coverage
pytest tests/test_auth.py       # Specific file
pytest -x                       # Stop at first failure
```

### Koniec materiałów teoretycznych!

**Gratulacje!** Przeszedłeś przez wszystkie 11 modułów FastAPI:
1. Routing
2. HTTP Mechanics
3. Pydantic Schemas
4. Request/Response
5. Dependency Injection
6. Database Setup
7. CRUD Operations
8. Relationships
9. Authentication
10. Authorization
11. Testing ✅